# APDTFlow Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yotambraun/APDTFlow/blob/main/Quickstart.ipynb)

**Forecast at any moment in time.** APDTFlow models time as continuous — one trained model answers three questions:

1. `predict()` — *what* will the value be on the forecast grid?
2. `predict_at(any timestamp)` — *what* at an arbitrary moment, including fractional steps and times beyond the trained horizon?
3. `predict_when(threshold)` — *when* will the series cross a level, with a calibrated time window?

This notebook trains one model on a real dataset (daily minimum temperatures, Melbourne) and asks it all three. Docs: [methodology](https://github.com/yotambraun/APDTFlow/blob/main/docs/METHODOLOGY.md) · [architecture](https://github.com/yotambraun/APDTFlow/blob/main/docs/models.md) · [measured results](https://github.com/yotambraun/APDTFlow/blob/main/docs/experiment_results.md)

In [ ]:
%pip install -q apdtflow

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import apdtflow
from apdtflow import APDTFlowForecaster

apdtflow.set_seed(0)  # bit-for-bit reproducible (deterministic algorithms, 1 CPU thread)
print('apdtflow', apdtflow.__version__)

In [ ]:
# Real data: daily minimum temperatures in Melbourne, 1981-1990.
URL = 'https://raw.githubusercontent.com/yotambraun/APDTFlow/main/dataset_examples/daily-minimum-temperatures-in-me_clean.csv'
LOCAL = 'dataset_examples/daily-minimum-temperatures-in-me_clean.csv'
try:
    df = pd.read_csv(URL)
except Exception:  # offline / local checkout
    df = pd.read_csv(LOCAL)

DATE, TARGET = 'Date', 'Daily minimum temperatures'
df[DATE] = pd.to_datetime(df[DATE])
print(df.shape)
df.tail(3)

## Train one model

Two flags unlock the continuous-time API:

- `decoder_type='continuous'` — the decoder integrates a small neural ODE forward in forecast time, so it can be queried at any real-valued offset (this powers `predict_at` / `predict_when`).
- `use_conformal=True` — calibrates distribution-free uncertainty during `fit` (required for `predict_when`). All uncertainty shown below comes from conformal calibration, never from the model's raw variance head.

In [ ]:
model = APDTFlowForecaster(
    forecast_horizon=14,        # trained grid: 14 days ahead
    history_length=60,          # each forecast looks at the last 60 days
    num_epochs=20,
    decoder_type='continuous',  # enables predict_at / predict_when
    use_conformal=True,         # calibrated uncertainty
)
model.fit(df, target_col=TARGET, date_col=DATE)

## Question 1 — `predict()`: the familiar grid forecast

In [ ]:
forecast = model.predict()  # 14 daily values

last_date = df[DATE].iloc[-1]
grid_dates = pd.date_range(last_date, periods=15, freq='D')[1:]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df[DATE].tail(90), df[TARGET].tail(90), color='gray', lw=1, label='history')
ax.plot(grid_dates, forecast, 'o-', color='C1', ms=4, label='14-day grid forecast')
ax.axvline(last_date, color='k', ls=':', lw=1)
ax.set_ylabel('daily min temperature (°C)')
ax.set_title('predict(): what will the value be?')
ax.legend()
plt.show()

## Question 2 — `predict_at()`: any moment in time

The decoder is continuous in forecast time, so you can query **between** days (36 hours out), at fractional offsets (7.25 days), and **beyond** the trained 14-day horizon — each with a conformal interval. Beyond the horizon, the last grid step's interval is reused (documented behavior, not a calibration guarantee out there).

In [ ]:
query_times = [
    last_date + pd.Timedelta(hours=36),   # one and a half days out (fractional step)
    last_date + pd.Timedelta(days=7.25),  # quarter past a week
    last_date + pd.Timedelta(days=16),    # beyond the trained 14-day horizon
]
values, lower, upper = model.predict_at(query_times)
for t, v, lo, hi in zip(query_times, values, lower, upper):
    print(f'{t}  ->  {v:5.2f} °C   (95% interval {lo:.2f} .. {hi:.2f})')

# A dense continuous trajectory: one model, evaluated at 150 arbitrary offsets.
dense_offsets = np.linspace(0.25, 18, 150)  # in days past the end of the data
dense, dense_lo, dense_hi = model.predict_at(dense_offsets)
dense_dates = [last_date + pd.Timedelta(days=float(o)) for o in dense_offsets]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df[DATE].tail(60), df[TARGET].tail(60), color='gray', lw=1, label='history')
ax.plot(dense_dates, dense, color='C0', lw=2, label='continuous forecast')
ax.fill_between(dense_dates, dense_lo, dense_hi, color='C0', alpha=0.2, label='95% conformal band')
ax.plot(grid_dates, forecast, 'o', color='C1', ms=4, label='grid forecast (predict)')
ax.axvline(grid_dates[-1], color='k', ls=':', lw=1, label='trained horizon')
ax.set_ylabel('daily min temperature (°C)')
ax.set_title('predict_at(): one model, any moment in time')
ax.legend(loc='upper left', fontsize=8)
plt.show()

## Question 3 — `predict_when()`: when does it cross the line?

`predict_when(threshold)` returns the first crossing of the forecast trajectory with a time window calibrated on **crossing-time errors** (not value bands mapped onto the time axis — that miscalibrates, as measured in the [methodology](https://github.com/yotambraun/APDTFlow/blob/main/docs/METHODOLOGY.md)).

> **The operational rule: schedule by `act_by` (the window's early edge), never by the point estimate `eta`.** The point estimate carries a systematic late bias on degradation data; the asymmetric calibration absorbs it.

Here we ask when daily minimums will next rise above a warm threshold (the data ends on 1990-12-31 — early Southern-Hemisphere summer, so temperatures are climbing). This demonstrates the API; the domains where `predict_when` has audited, baseline-beating skill are degradation and depletion problems — see the [battery RUL playground](https://github.com/yotambraun/APDTFlow/blob/main/examples/battery_rul_playground.ipynb).

In [ ]:
# A warm threshold chosen from the data: the 90th percentile of the last year.
threshold = round(float(df[TARGET].tail(365).quantile(0.90)), 1)

result = model.predict_when(threshold=threshold, direction='above', alpha=0.1)
print(f'When will the daily minimum rise above {threshold} °C?')
print(f'  eta (point estimate): {result.eta}')
print(f'  90% time window:      {result.earliest}  ..  {result.latest}')
print(f'  act_by:               {result.act_by}   <- schedule by this, never by eta')
print(f'  censored:             {result.censored}  (True = no crossing within the horizon — a valid answer)')

## Calibrated intervals on the grid forecast

`use_conformal=True` also gives the plain grid forecast distribution-free intervals via `predict(return_intervals='conformal')`.

In [ ]:
lo, preds, hi = model.predict(return_intervals='conformal', alpha=0.05)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df[DATE].tail(45), df[TARGET].tail(45), color='gray', lw=1, label='history')
ax.plot(grid_dates, preds, 'o-', color='C1', ms=4, label='forecast')
ax.fill_between(grid_dates, lo, hi, color='C1', alpha=0.25, label='95% conformal interval')
ax.axvline(last_date, color='k', ls=':', lw=1)
ax.set_ylabel('daily min temperature (°C)')
ax.set_title('Conformal prediction intervals (split conformal, alpha=0.05)')
ax.legend()
plt.show()

## Where to go next

- **Operational rule**, once more: act on `result.act_by`, never on `eta`. `alpha` is the sensitivity knob — smaller alpha means wider windows and earlier act-by dates.
- [examples/](https://github.com/yotambraun/APDTFlow/tree/main/examples) — `predict_at_demo.py`, `predict_when_demo.py`, the battery RUL playground, exogenous/categorical features, backtesting, FastAPI serving.
- [docs/METHODOLOGY.md](https://github.com/yotambraun/APDTFlow/blob/main/docs/METHODOLOGY.md) — how everything works, the evaluation protocol, and what we tested and chose **not** to ship.
- [docs/models.md](https://github.com/yotambraun/APDTFlow/blob/main/docs/models.md) — the architecture and parameter reference.
- [docs/experiment_results.md](https://github.com/yotambraun/APDTFlow/blob/main/docs/experiment_results.md) — measured results, all reproducible from committed scripts.
- Fleet maintenance: `model.predict_when_fleet({asset_id: series, ...}, threshold, direction)` returns a schedule sorted by `act_by`.